# NHL Game Data — RL Notebook (Fixed)

> ✅ **How to run this notebook:**
> 1. Attach your Lakehouse (left panel → Add lakehouse)
> 2. Click **Run all**
> 3. That's it — no installs, no kernel restarts needed

---
## Step 1 — Imports & Session Config

In [ ]:
import os
import pandas as pd
from pyspark.sql.functions import col, to_date, sum, count, round, when

# Allow Delta schema changes for the whole session
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("Session config set.")
print("Spark version:", spark.version)

---
## Step 2 — Locate Dataset

Automatically checks the Kaggle cache from RL's previous run.
If not found, falls back to Lakehouse Files.

In [ ]:
# Try cache path first (from RL's previous kagglehub download)
CACHE_PATH    = "/home/trusted-service-user/.cache/kagglehub/datasets/martinellis/nhl-game-data/versions/2"
LAKEHOUSE_PATH = "/lakehouse/default/Files/nhl_raw"

if os.path.exists(CACHE_PATH) and len(os.listdir(CACHE_PATH)) > 0:
    path = CACHE_PATH
    print(f"Using cached dataset: {path}")
elif os.path.exists(LAKEHOUSE_PATH) and len(os.listdir(LAKEHOUSE_PATH)) > 0:
    path = LAKEHOUSE_PATH
    print(f"Using Lakehouse Files: {path}")
else:
    raise FileNotFoundError(
        "Dataset not found in cache or Lakehouse Files.\n"
        "Please upload the NHL CSVs to Files/nhl_raw/ in your Lakehouse."
    )

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
print(f"Found {len(csv_files)} CSV files: {csv_files}")

---
## Step 3 — Load Raw CSVs into Bronze Tables

In [ ]:
for file in csv_files:
    file_path  = f"{path}/{file}"
    table_name = file.replace(".csv", "").lower()

    print(f"Loading {file} -> {table_name}")
    pdf = pd.read_csv(file_path)
    sdf = spark.createDataFrame(pdf)

    sdf.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

    print(f"  Saved {table_name}: {pdf.shape[0]:,} rows")

print("\nAll CSVs loaded successfully.")

---
## Step 4 — Inspect Tables & Row Counts

In [ ]:
for table in spark.sql("SHOW TABLES").select("tableName").collect():
    t = table["tableName"]
    print(f"{t}: {spark.table(t).count():,} rows")

---
## Step 5 — Preview Key Tables

In [ ]:
display(spark.table("game").limit(5))
display(spark.table("team_info").limit(5))
display(spark.table("player_info").limit(5))
display(spark.table("game_teams_stats").limit(5))
display(spark.table("game_skater_stats").limit(5))
display(spark.table("game_plays").limit(5))

---
## Step 6 — Silver Layer: Clean & Deduplicate

In [ ]:
# silver_game
silver_game = (
    spark.table("game")
    .dropDuplicates(["game_id"])
    .withColumn("date_time_gmt", col("date_time_gmt").cast("timestamp"))
    .withColumn("game_date", to_date(col("date_time_gmt")))
)
silver_game.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_game")
print(f"silver_game: {silver_game.count():,} rows")

In [ ]:
# silver_team
silver_team = spark.table("team_info").dropDuplicates(["team_id"])
silver_team.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_team")
print(f"silver_team: {silver_team.count()} rows")

In [ ]:
# silver_player
silver_player = spark.table("player_info").dropDuplicates(["player_id"])
silver_player.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_player")
print(f"silver_player: {silver_player.count():,} rows")

In [ ]:
# silver_team_game_stats
silver_team_game_stats = (
    spark.table("game_teams_stats").dropDuplicates(["game_id", "team_id"])
)
silver_team_game_stats.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_team_game_stats")
print(f"silver_team_game_stats: {silver_team_game_stats.count():,} rows")

In [ ]:
# silver_game_skater_stats
silver_game_skater_stats = (
    spark.table("game_skater_stats")
    .dropDuplicates(["game_id", "player_id"])
    .withColumn("shooting_pct",
        round(when(col("shots") > 0, col("goals") / col("shots") * 100).otherwise(0), 2))
    .withColumn("points", col("goals") + col("assists"))
    .withColumn("toi_minutes", round(col("timeOnIce") / 60, 2))
    .na.fill(0, subset=["goals", "assists", "shots", "hits", "blocked",
                        "penaltyMinutes", "giveaways", "takeaways", "plusMinus"])
)
silver_game_skater_stats.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_game_skater_stats")
print(f"silver_game_skater_stats: {silver_game_skater_stats.count():,} rows")

---
## Step 7 — Data Validation

In [ ]:
print("--- Duplicate game IDs ---")
spark.sql("""
    SELECT game_id, COUNT(*) AS count FROM silver_game
    GROUP BY game_id HAVING COUNT(*) > 1
""").show()

print("--- Null checks on silver_game ---")
spark.sql("""
    SELECT
        SUM(CASE WHEN game_id IS NULL THEN 1 ELSE 0 END)      AS null_game_ids,
        SUM(CASE WHEN home_team_id IS NULL THEN 1 ELSE 0 END) AS null_home_teams,
        SUM(CASE WHEN away_team_id IS NULL THEN 1 ELSE 0 END) AS null_away_teams
    FROM silver_game
""").show()

print("--- Orphan team IDs in team stats ---")
spark.sql("""
    SELECT t.team_id, COUNT(*) AS missing_count
    FROM silver_team_game_stats t
    LEFT JOIN silver_team s ON t.team_id = s.team_id
    WHERE s.team_id IS NULL
    GROUP BY t.team_id ORDER BY missing_count DESC
""").show()

---
## Step 8 — Remove Orphan Records & Create Validated Silver Table

In [ ]:
# Inner join removes 10 orphan records (team_ids 87-90 with no team reference)
silver_team_game_stats_valid = (
    spark.table("silver_team_game_stats").alias("t")
    .join(spark.table("silver_team").alias("s"),
          col("t.team_id") == col("s.team_id"), "inner")
    .select("t.*")
)
silver_team_game_stats_valid.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_team_game_stats_valid")
print(f"silver_team_game_stats_valid: {silver_team_game_stats_valid.count():,} rows")

---
## Step 9 — Gold Layer: Team Performance

In [ ]:
gold_team_performance = (
    spark.table("silver_team_game_stats_valid")
    .groupBy("team_id")
    .agg(
        count("game_id").alias("games_played"),
        sum(col("won").cast("int")).alias("wins"),
        sum("goals").alias("goals_for"),
        sum("shots").alias("shots_for"),
        sum("hits").alias("total_hits"),
        sum("pim").alias("penalty_minutes"),
        sum("powerPlayGoals").alias("power_play_goals"),
        sum("powerPlayOpportunities").alias("power_play_opportunities"),
        sum("giveaways").alias("giveaways"),
        sum("takeaways").alias("takeaways")
    )
    .withColumn("losses", col("games_played") - col("wins"))
    .withColumn("win_percentage",
        round((col("wins") / col("games_played")) * 100, 2))
    .withColumn("avg_goals_per_game",
        round(col("goals_for") / col("games_played"), 2))
    .withColumn("avg_shots_per_game",
        round(col("shots_for") / col("games_played"), 2))
    .withColumn("power_play_success_rate",
        round(when(col("power_play_opportunities") > 0,
                   col("power_play_goals") / col("power_play_opportunities") * 100)
              .otherwise(0), 2))
)
gold_team_performance.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_team_performance")
print(f"gold_team_performance: {gold_team_performance.count()} rows")

In [ ]:
gold_team_performance_named = (
    spark.table("gold_team_performance").alias("g")
    .join(spark.table("silver_team").alias("t"),
          col("g.team_id") == col("t.team_id"), "left")
    .select(
        col("g.team_id"),
        col("t.shortName").alias("team_location"),
        col("t.teamName").alias("team_name"),
        col("t.abbreviation"),
        col("g.games_played"), col("g.wins"), col("g.losses"),
        col("g.goals_for"), col("g.shots_for"), col("g.total_hits"),
        col("g.penalty_minutes"), col("g.power_play_goals"),
        col("g.power_play_opportunities"), col("g.giveaways"), col("g.takeaways"),
        col("g.win_percentage"), col("g.avg_goals_per_game"),
        col("g.avg_shots_per_game"), col("g.power_play_success_rate")
    )
)
gold_team_performance_named.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_team_performance_named")
print(f"gold_team_performance_named: {gold_team_performance_named.count()} rows")
display(gold_team_performance_named.orderBy(col("win_percentage").desc()))

---
## Step 10 — Gold Layer: Player Performance

In [ ]:
gold_player_performance = (
    spark.table("silver_player").alias("p")
    .join(spark.table("silver_game_skater_stats").alias("s"),
          col("p.player_id") == col("s.player_id"), "inner")
    .groupBy(
        col("p.player_id"), col("p.firstName"), col("p.lastName"),
        col("p.nationality"), col("p.primaryPosition")
    )
    .agg(
        count("s.game_id").alias("games_played"),
        sum("s.goals").alias("goals"),
        sum("s.assists").alias("assists"),
        sum("s.shots").alias("shots"),
        sum("s.hits").alias("hits"),
        sum("s.penaltyMinutes").alias("penalty_minutes"),
        sum("s.powerPlayGoals").alias("power_play_goals"),
        sum("s.powerPlayAssists").alias("power_play_assists"),
        sum("s.takeaways").alias("takeaways"),
        sum("s.giveaways").alias("giveaways"),
        sum("s.blocked").alias("blocked_shots")
    )
    .withColumn("points", col("goals") + col("assists"))
    .withColumn("avg_points_per_game",
        round(col("points") / col("games_played"), 2))
    .withColumn("shot_conversion_rate",
        round(when(col("shots") > 0, col("goals") / col("shots") * 100)
              .otherwise(None), 2))
)
gold_player_performance.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_player_performance")
print(f"gold_player_performance: {gold_player_performance.count():,} rows")
display(gold_player_performance.orderBy(col("points").desc()))

---
## Step 11 — Gold Layer: Game Summary

In [ ]:
gold_game_summary = (
    spark.table("silver_game").alias("g")
    .join(spark.table("silver_team").alias("home"),
          col("g.home_team_id") == col("home.team_id"), "left")
    .join(spark.table("silver_team").alias("away"),
          col("g.away_team_id") == col("away.team_id"), "left")
    .select(
        col("g.game_id"), col("g.season"), col("g.type"),
        col("g.game_date"), col("g.date_time_gmt"),
        col("home.teamName").alias("home_team"),
        col("away.teamName").alias("away_team"),
        col("g.home_goals"), col("g.away_goals"),
        col("g.outcome"), col("g.venue")
    )
)
gold_game_summary.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_game_summary")
print(f"gold_game_summary: {gold_game_summary.count():,} rows")
display(gold_game_summary)

---
## Step 12 — Confirm All Tables

In [ ]:
print("=== Silver Tables ===")
spark.sql("SHOW TABLES LIKE 'silver*'").show(truncate=False)

print("=== Gold Tables ===")
spark.sql("SHOW TABLES LIKE 'gold*'").show(truncate=False)

### 📌 IMPORTANT: Star Schema for your Semantic Model

This notebook (`RL_Notebook_Fixed`) produces:
- Bronze layer (raw tables like `game`, `team_info`, `player_info`, etc.)
- Silver layer + some `silver_*` cleaned tables (`silver_team_game_stats_valid` etc.)
- Legacy non-dimensional gold tables: `gold_team_performance`, `gold_player_performance`, `gold_game_summary`

**The Power BI semantic model shown in your diagram (and documented in schema/star_schema.md + dashboard/power_bi_guide.md) depends on the DIMENSIONAL model and newer gold aggregates:**

`dim_team`, `dim_player`, `dim_date`
`fact_game_performance`, `fact_player_stats`, `fact_play_events`
`gold_team_standings`, `gold_player_season_stats`, `gold_home_away_summary`, `gold_goals_by_period`, `gold_powerplay_efficiency`

➡️ These tables are created by: **`notebooks/04_Gap_Fill_StarSchema_and_Testing.ipynb`**

Run order for the full end-to-end including the semantic model:
1. This notebook (RL_Notebook_Fixed) — for bronze/silver basics
2. `04_Gap_Fill_StarSchema_and_Testing.ipynb` — adds the star schema matching your attached model screenshot

After both, connect in Power BI using DirectLake to the Gold star tables (see power_bi_guide.md). The 04 notebook also runs built-in validation for referential integrity etc.

See updated README.md and docs/architecture.md for details.
